# PullRequestReviewEvent pipeline test (`05_pull_request_review_events`)

This notebook tests `05_pull_request_review_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_review`. It runs the real notebook with `%run` and `test_schema = test_review`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `05_pull_request_review_events` (they create `silver.pull_request_review_events` and `quarantine.quarantine_pull_request_review_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to PullRequestReviewEvent: the five review fields are read from `payload.pull_request` and `payload.review` and converted, `review_state` is stored exactly as received (never lower-cased or replaced), all five profiled states pass without a warning, an unexpected action or state is kept with an `UNEXPECTED_VALUE` warning, every problem in a review field is kept with a warning instead of quarantined, the same event appearing three times (as in the real data) becomes one row, and the quarantine table keeps the review fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_review"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("pull_request_review_events")
TEST_QUARANTINE = T("quarantine_pull_request_review_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.pull_request_review_events",
    "github_lakehouse.quarantine.quarantine_pull_request_review_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 05_pull_request_review_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.pull_request_review_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_pull_request_review_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default event is a `created` review `id 8001` with state `approved` on pull request `id 9001`, `number 42`.

| Row | Case | Expected result |
|---|---|---|
| `v1` | clean PullRequestReviewEvent | Silver, every field converted, no warning |
| `v2` | `action` missing | Silver, `review_action` NULL, warning `MISSING: review_action` |
| `v3` | `action = submitted` | Silver, value kept, warning `UNEXPECTED_VALUE: review_action` |
| `v4` | `review.state` missing | Silver, `review_state` NULL (not `commented`), warning `MISSING: review_state` |
| `v5` | `review.state = APPROVED` | Silver, value kept as `APPROVED` (not lower-cased), warning `UNEXPECTED_VALUE: review_state` |
| `v6` | `review.state` is blank | Silver, `review_state` NULL, warning `MISSING: review_state` |
| `v7` | `review.id = "abc"` | Silver, `review_id` NULL, warning `INVALID_CAST: review_id` |
| `v8` | no `pull_request` object | Silver, two `MISSING` warnings, counted in `null_pr_id_and_number` |
| `v9a`–`v9d` | states `commented`, `changes_requested`, `dismissed`, `pending` | Silver, no warning |
| `v10` | `review` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `v11` | `repo.id = "x"` | quarantine `INVALID_CAST: repo_id`, review fields and original text kept |
| `v12` (3 times) | the same event, identical JSON: once in an older file, twice in a newer file | one Silver row, from the newer file |
| `x1` | a PullRequestReviewCommentEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

MISSING = object()   # marker: leave the key out of the JSON

def payload(action="created", pull_request=None, review=None):
    """A review payload. Defaults = a clean, created, approved review."""
    p = {
        "action": action,
        "pull_request": {"id": 9001, "number": 42} if pull_request is None else pull_request,
        "review": {"id": 8001, "state": "approved"} if review is None else review,
    }
    return {k: v for k, v in p.items() if v is not MISSING}

def ev(event_id, type_="PullRequestReviewEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, payload_=None):
    """One GitHub-style event as a JSON string. Defaults = a clean event."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": payload() if payload_ is None else payload_,
    }
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                        # case
    (ev("v1"),                                                      F1, D1, I1),   # clean
    (ev("v2", payload_=payload(action=MISSING)),                    F1, D1, I1),   # action missing
    (ev("v3", payload_=payload(action="submitted")),                F1, D1, I1),   # unexpected action
    (ev("v4", payload_=payload(review={"id": 8001})),               F1, D1, I1),   # state missing
    (ev("v5", payload_=payload(review={"id": 8001, "state": "APPROVED"})),
                                                                    F1, D1, I1),   # unexpected state
    (ev("v6", payload_=payload(review={"id": 8001, "state": "  "})),
                                                                    F1, D1, I1),   # blank state
    (ev("v7", payload_=payload(review={"id": "abc", "state": "approved"})),
                                                                    F1, D1, I1),   # review_id invalid
    (ev("v8", payload_=payload(pull_request=MISSING)),              F1, D1, I1),   # no pull_request
    (ev("v9a", payload_=payload(review={"id": 8001, "state": "commented"})),         F1, D1, I1),
    (ev("v9b", payload_=payload(review={"id": 8001, "state": "changes_requested"})), F1, D1, I1),
    (ev("v9c", payload_=payload(review={"id": 8001, "state": "dismissed"})),         F1, D1, I1),
    (ev("v9d", payload_=payload(review={"id": 8001, "state": "pending"})),           F1, D1, I1),
    (ev("v10", payload_=payload(review="oops")),                    F1, D1, I1),   # parse error
    (ev("v11", repo={"id": "x", "name": "octo/repo"}),              F1, D1, I1),   # repo_id invalid
    (ev("v12"),                                                     F1, D1, I1),   # duplicate, older
    (ev("v12"),                                                     F2, D1, I2),   # duplicate, newer -> wins
    (ev("v12"),                                                     F2, D1, I2),   # exact copy of the newer line
    (ev("x1", type_="PullRequestReviewCommentEvent"),               F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 18)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../05_pull_request_review_events $test_schema="test_review" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_review_action": 1, "null_pr_id": 1, "null_pr_number": 1,
    "null_pr_id_and_number": 1, "null_review_id": 1, "null_review_state": 2,
    "unexpected_review_action": 1, "unexpected_review_state": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_pull_request_review_events")
check("audit event_type", audit["event_type"], "PullRequestReviewEvent")

# --- audit: 17 PullRequestReviewEvent rows (x1 is another type)
check_audit({
    "bronze_input_rows": 17, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 15,
    "duplicate_rows_removed": 2, "silver_candidate_rows": 13,
    "silver_rows_inserted": 13, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, event fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "review_action", "pr_id", "pr_number", "review_id", "review_state",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      sorted(["v1", "v2", "v3", "v4", "v5", "v6", "v7", "v8",
              "v9a", "v9b", "v9c", "v9d", "v12"]))

def review_fields(r):
    return (r.get("review_action"), r.get("pr_id"), r.get("pr_number"),
            r.get("review_id"), r.get("review_state"))

r = one_silver("v1")
check("v1 fields", review_fields(r), ("created", 9001, 42, 8001, "approved"))
check("v1 common fields", (r.get("actor_id"), r.get("repo_id"), str(r.get("event_date"))),
      (101, 201, "2024-02-01"))
check("v1 warning", r.get("quarantine_warning"), None)

r = one_silver("v2")
check("v2 action", r.get("review_action"), None)
check("v2 warning", r.get("quarantine_warning"), "MISSING: review_action")

r = one_silver("v3")
check("v3 action kept unchanged", r.get("review_action"), "submitted")
check("v3 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: review_action")

r = one_silver("v4")
check("v4 state NULL, not replaced", r.get("review_state"), None)
check("v4 warning", r.get("quarantine_warning"), "MISSING: review_state")

r = one_silver("v5")
check("v5 state kept exactly", r.get("review_state"), "APPROVED")
check("v5 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: review_state")

r = one_silver("v6")
check("v6 blank state -> NULL", r.get("review_state"), None)
check("v6 warning", r.get("quarantine_warning"), "MISSING: review_state")

r = one_silver("v7")
check("v7 review_id", r.get("review_id"), None)
check("v7 warning", r.get("quarantine_warning"), "INVALID_CAST: review_id")

r = one_silver("v8")
check("v8 fields", review_fields(r), ("created", None, None, 8001, "approved"))
check("v8 warnings", r.get("quarantine_warning"), "MISSING: pr_id; MISSING: pr_number")

for event_id, state in [("v9a", "commented"), ("v9b", "changes_requested"),
                        ("v9d", "pending"), ("v9c", "dismissed")]:
    r = one_silver(event_id)
    check(f"{event_id} state", r.get("review_state"), state)
    check(f"{event_id} no warning", r.get("quarantine_warning"), None)

r = one_silver("v12")
check("v12 winner = newer file", r.get("_source_file"), F2)

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["INVALID_CAST: repo_id", "JSON_PARSE_ERROR"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("v10 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["v10"])
v11 = [r for r in q if r["event_id"] == "v11"]
check("v11 found once", len(v11), 1)
r = v11[0] if v11 else {}
check("v11 reason", r.get("quarantine_reason"), "INVALID_CAST: repo_id")
check("v11 fields kept", review_fields(r), ("created", 9001, 42, 8001, "approved"))
check("v11 original text kept",
      (r.get("pr_id_raw"), r.get("pr_number_raw"), r.get("review_id_raw"), r.get("repo_id_raw")),
      ("9001", "42", "8001", "x"))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../05_pull_request_review_events $test_schema="test_review" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 17, "silver_candidate_rows": 13,
    "silver_rows_inserted": 0, "silver_rows_already_present": 13,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 13)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")